In [ ]:
# %%
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
DATA_DIR = PROJECT_ROOT / "data"

main_frame = pd.read_csv(DATA_DIR / "daily_variance_measures_filtered.csv", parse_dates=["date"])

print("Loaded:", DATA_DIR / "daily_variance_measures_filtered.csv")
print("Shape:", main_frame.shape)
print(main_frame.head())

Loaded: c:\Users\synnerbo\Code\tio4900_master_thesis\data\daily_variance_measures_filtered.csv
Shape: (3718, 33)
        date    RV_dec    RV_pct     close  RVOL_pct_ann    CC_dec    CC_pct  \
0 2008-02-01  0.000044  0.441615  1.482225     10.549260  0.000044  0.443001   
1 2008-02-04  0.000014  0.144456  1.481392      6.033483  0.000012  0.115579   
2 2008-02-05  0.000032  0.315997  1.464040      8.923629  0.000030  0.297090   
3 2008-02-06  0.000031  0.309158  1.465788      8.826538  0.000024  0.242737   
4 2008-02-07  0.000053  0.525159  1.450133     11.503918  0.000029  0.292443   

     JC_dec    JC_pct         MedRQ  ...  log_return    return    RV_vol  \
0  0.000000  0.000000  9.827328e-09  ...   -0.000562 -0.000562  0.006645   
1  0.000003  0.028877  3.045381e-10  ...   -0.011783 -0.011714  0.003801   
2  0.000002  0.018906  2.060522e-09  ...    0.001193  0.001194  0.005621   
3  0.000007  0.066421  9.736315e-10  ...   -0.010738 -0.010680  0.005560   
4  0.000023  0.232716  2.1

In [36]:
EPS = 1e-12

# --- Load raw IV CSV ---
iv_raw = pd.read_csv(DATA_DIR / "IV_surface_raw.csv")

# Parse IV date (it is like 20070102)
iv_raw["date"] = pd.to_datetime(iv_raw["date"].astype(str), format="%Y%m%d", errors="coerce")

# ATM IV inputs
atm_iv_cols = [
    "D_1_Put_50", "D_1_Call_50",
    "W_1_Put_50", "W_1_Call_50",
    "M_1_Put_50", "M_1_Call_50",
    "M_3_Put_50", "M_3_Call_50",
    "M_6_Put_50", "M_6_Call_50",
    "Y_1_Put_50", "Y_1_Call_50",
]

# RR/BF inputs
rr_bf_cols = [
    "D_1_RR_25", "W_1_RR_25", "M_1_RR_25",
    "D_1_BF_25", "W_1_BF_25", "M_1_BF_25",
]

all_surface_cols = atm_iv_cols + rr_bf_cols


missing = [c for c in all_surface_cols if c not in iv_raw.columns]
if missing:
    raise ValueError(f"Missing expected surface columns: {missing}")

iv = iv_raw[["date"] + all_surface_cols].copy()


for c in all_surface_cols:
    iv[c] = pd.to_numeric(iv[c], errors="coerce")

# --- Align to trading calendar (same dates as RV) ---
dates = pd.read_csv(DATA_DIR / "dates_only.csv")
dates["date"] = pd.to_datetime(dates["datetime"])
dates = dates[["date"]].drop_duplicates().sort_values("date")

iv = dates.merge(iv, on="date", how="left")

print("Before IV dropna:", iv.shape)
print("Missing share before drop:")
print(iv[all_surface_cols].isna().mean().sort_values(ascending=False))

iv = iv.dropna(subset=all_surface_cols, how="any").copy()

print("After IV dropna:", iv.shape)

# Scale ATM IV inputs
for c in atm_iv_cols:
    iv[f"IVvol_daily_{c}"] = (iv[c] / 100.0) / np.sqrt(252.0)

# Scale RR and BF inputs
for c in rr_bf_cols:
    iv[f"{c}_daily"] = (iv[c] / 100.0) / np.sqrt(252.0)


# Create ATM average volatility per maturity
iv["IV_D1"] = (
    iv["IVvol_daily_D_1_Put_50"] +
    iv["IVvol_daily_D_1_Call_50"]
) / 2

iv["IV_W1"] = (
    iv["IVvol_daily_W_1_Put_50"] +
    iv["IVvol_daily_W_1_Call_50"]
) / 2

iv["IV_M1"] = (
    iv["IVvol_daily_M_1_Put_50"] +
    iv["IVvol_daily_M_1_Call_50"]
) / 2

iv["IV_M3"] = (
    iv["IVvol_daily_M_3_Put_50"] +
    iv["IVvol_daily_M_3_Call_50"]
) / 2

iv["IV_M6"] = (
    iv["IVvol_daily_M_6_Put_50"] +
    iv["IVvol_daily_M_6_Call_50"]
) / 2

iv["IV_Y1"] = (
    iv["IVvol_daily_Y_1_Put_50"] +
    iv["IVvol_daily_Y_1_Call_50"]
) / 2

# --- Term structure slope features ---
iv["IV_SLOPE_D1_W1"] = iv["IV_W1"] - iv["IV_D1"]
iv["IV_SLOPE_D1_M1"] = iv["IV_M1"] - iv["IV_D1"]
iv["IV_SLOPE_W1_M1"] = iv["IV_M1"] - iv["IV_W1"]
iv["IV_SLOPE_M1_M3"] = iv["IV_M3"] - iv["IV_M1"]
iv["IV_SLOPE_M3_M6"] = iv["IV_M6"] - iv["IV_M3"]
iv["IV_SLOPE_M1_Y1"] = iv["IV_Y1"] - iv["IV_M1"]
iv["IV_SLOPE_M6_Y1"] = iv["IV_Y1"] - iv["IV_M6"]

# --- Term structure curvature features ---
iv["IV_CURVE_D1_W1_M1"] = iv["IV_M1"] - 2 * iv["IV_W1"] + iv["IV_D1"]
iv["IV_CURVE_W1_M1_M3"] = iv["IV_M3"] - 2 * iv["IV_M1"] + iv["IV_W1"]
iv["IV_CURVE_M1_M6_Y1"] = iv["IV_Y1"] - 2 * iv["IV_M6"] + iv["IV_M1"]
iv["IV_CURVE_M3_M6_Y1"] = iv["IV_Y1"] - 2 * iv["IV_M6"] + iv["IV_M3"]

# --- Risk reversal features ---
iv["RR_D1"] = iv["D_1_RR_25_daily"]
iv["RR_W1"] = iv["W_1_RR_25_daily"]
iv["RR_M1"] = iv["M_1_RR_25_daily"]

# --- Butterfly features ---
iv["BF_D1"] = iv["D_1_BF_25_daily"]
iv["BF_W1"] = iv["W_1_BF_25_daily"]
iv["BF_M1"] = iv["M_1_BF_25_daily"]

iv_clean = iv[
    [
        "date",
        "IV_D1", "IV_W1", "IV_M1", "IV_M3", "IV_M6", "IV_Y1",
        "IV_SLOPE_D1_W1", "IV_SLOPE_D1_M1", "IV_SLOPE_W1_M1", "IV_SLOPE_M1_M3", "IV_SLOPE_M3_M6", "IV_SLOPE_M1_Y1","IV_SLOPE_M6_Y1",
        "IV_CURVE_D1_W1_M1", "IV_CURVE_W1_M1_M3", "IV_CURVE_M1_M6_Y1","IV_CURVE_M3_M6_Y1",
        "RR_D1", "RR_W1", "RR_M1",
        "BF_D1", "BF_W1", "BF_M1",
    ]
].copy()

#iv_clean.to_csv(DATA_DIR / "IV_ATM_daily_vol_dec.csv", index=False)

print(iv_clean.head())
print(iv_clean.isna().mean())

Before IV dropna: (3741, 19)
Missing share before drop:
M_3_Put_50     0.001337
M_6_Put_50     0.001337
M_1_BF_25      0.001069
Y_1_Put_50     0.001069
M_1_Put_50     0.001069
D_1_Put_50     0.000802
D_1_Call_50    0.000802
W_1_Call_50    0.000802
M_1_Call_50    0.000802
W_1_Put_50     0.000802
M_6_Call_50    0.000802
M_3_Call_50    0.000802
D_1_RR_25      0.000802
Y_1_Call_50    0.000802
W_1_RR_25      0.000802
M_1_RR_25      0.000802
D_1_BF_25      0.000802
W_1_BF_25      0.000802
dtype: float64
After IV dropna: (3733, 19)
        date     IV_D1     IV_W1     IV_M1     IV_M3     IV_M6     IV_Y1  \
0 2008-01-02  0.006614  0.006221  0.006157  0.005699  0.005477  0.005292   
1 2008-01-03  0.009453  0.006252  0.006078  0.005809  0.005509  0.005339   
2 2008-01-04  0.003467  0.006051  0.005938  0.005699  0.005554  0.005354   
3 2008-01-07  0.007874  0.005827  0.006126  0.005778  0.005524  0.005417   
4 2008-01-08  0.007874  0.005512  0.005693  0.005558  0.005414  0.005276   

   IV_SLOPE_

In [37]:
main_frame["date"] = pd.to_datetime(main_frame["date"])
iv_clean["date"] = pd.to_datetime(iv_clean["date"])

joint_ext = main_frame.merge(iv_clean, on="date", how="inner")

print("joint_ext shape:", joint_ext.shape)
print("date range:", joint_ext["date"].min(), "to", joint_ext["date"].max())

print("Missing shares:")
print(joint_ext[[
    "IV_D1", "IV_W1", "IV_M1", "IV_M3", "IV_M6", "IV_Y1",
    "IV_SLOPE_D1_W1", "IV_SLOPE_D1_M1", "IV_SLOPE_W1_M1", "IV_SLOPE_M1_M3", "IV_SLOPE_M3_M6", "IV_SLOPE_M1_Y1","IV_SLOPE_M6_Y1",
    "IV_CURVE_D1_W1_M1", "IV_CURVE_W1_M1_M3", "IV_CURVE_M1_M6_Y1","IV_CURVE_M3_M6_Y1",
    "RR_D1", "RR_W1", "RR_M1",
    "BF_D1", "BF_W1", "BF_M1"
]].isna().mean())

#out_fp = DATA_DIR / "RV_IV_extended.csv"
#joint_ext.to_csv(out_fp, index=False)



joint_ext shape: (3710, 56)
date range: 2008-02-01 00:00:00 to 2022-12-29 00:00:00
Missing shares:
IV_D1                0.0
IV_W1                0.0
IV_M1                0.0
IV_M3                0.0
IV_M6                0.0
IV_Y1                0.0
IV_SLOPE_D1_W1       0.0
IV_SLOPE_D1_M1       0.0
IV_SLOPE_W1_M1       0.0
IV_SLOPE_M1_M3       0.0
IV_SLOPE_M3_M6       0.0
IV_SLOPE_M1_Y1       0.0
IV_SLOPE_M6_Y1       0.0
IV_CURVE_D1_W1_M1    0.0
IV_CURVE_W1_M1_M3    0.0
IV_CURVE_M1_M6_Y1    0.0
IV_CURVE_M3_M6_Y1    0.0
RR_D1                0.0
RR_W1                0.0
RR_M1                0.0
BF_D1                0.0
BF_W1                0.0
BF_M1                0.0
dtype: float64


In [38]:
# %%
# Create extended combo1-style dataset
combo1_ext = joint_ext.copy()

rename_map = {
    "RV_dec": "RV",
    "RV_W_dec": "RV_W",
    "RV_M_dec": "RV_M",
    "CC_dec": "CC",
    "JC_dec": "JC",
    "PV_dec": "PV",
    "NV_dec": "NV",
    "SJ_dec": "SJ",
    "CQ": "CQ",
}

combo1_ext = combo1_ext.rename(columns=rename_map)

keep_cols = [
    "date", "close", "log_return", "log_return_unshifted", "return", "return_unshifted",
    "RV", "RV_W", "RV_M",
    "CC", "JC", "PV", "NV", "SJ",
    "CQ",
    "IV_D1", "IV_W1", "IV_M1", "IV_M3", "IV_M6", "IV_Y1",
    "IV_SLOPE_D1_W1", "IV_SLOPE_D1_M1", "IV_SLOPE_W1_M1", "IV_SLOPE_M1_M3", "IV_SLOPE_M3_M6", "IV_SLOPE_M1_Y1","IV_SLOPE_M6_Y1",
    "IV_CURVE_D1_W1_M1", "IV_CURVE_W1_M1_M3", "IV_CURVE_M1_M6_Y1","IV_CURVE_M3_M6_Y1",
    "RR_D1", "RR_W1", "RR_M1",
    "BF_D1", "BF_W1", "BF_M1",
]

missing = [c for c in keep_cols if c not in combo1_ext.columns]
if missing:
    raise ValueError(f"Missing columns in combo1_ext: {missing}")

combo1_ext = combo1_ext[keep_cols].copy()

out_fp = DATA_DIR / "RV_IV_extended_slope_curve.csv"
combo1_ext.to_csv(out_fp, index=False)

print("Saved:", out_fp)
print("Shape:", combo1_ext.shape)
print("Columns:", combo1_ext.columns.tolist())
print(combo1_ext.head())

Saved: c:\Users\synnerbo\Code\tio4900_master_thesis\data\RV_IV_extended_slope_curve.csv
Shape: (3710, 38)
Columns: ['date', 'close', 'log_return', 'log_return_unshifted', 'return', 'return_unshifted', 'RV', 'RV_W', 'RV_M', 'CC', 'JC', 'PV', 'NV', 'SJ', 'CQ', 'IV_D1', 'IV_W1', 'IV_M1', 'IV_M3', 'IV_M6', 'IV_Y1', 'IV_SLOPE_D1_W1', 'IV_SLOPE_D1_M1', 'IV_SLOPE_W1_M1', 'IV_SLOPE_M1_M3', 'IV_SLOPE_M3_M6', 'IV_SLOPE_M1_Y1', 'IV_SLOPE_M6_Y1', 'IV_CURVE_D1_W1_M1', 'IV_CURVE_W1_M1_M3', 'IV_CURVE_M1_M6_Y1', 'IV_CURVE_M3_M6_Y1', 'RR_D1', 'RR_W1', 'RR_M1', 'BF_D1', 'BF_W1', 'BF_M1']
        date     close  log_return  log_return_unshifted    return  \
0 2008-02-01  1.482225   -0.000562              0.001114 -0.000562   
1 2008-02-04  1.481392   -0.011783             -0.000562 -0.011714   
2 2008-02-05  1.464040    0.001193             -0.011783  0.001194   
3 2008-02-06  1.465788   -0.010738              0.001193 -0.010680   
4 2008-02-07  1.450133   -0.000419             -0.010738 -0.000419   

  

In [39]:
combo1_ext[["BF_D1","BF_W1","BF_M1"]].describe()

,BF_D1,BF_W1,BF_M1
count,3710.000000,3710.000000,3710.000000
mean,0.000165,0.000128,0.000144
std,0.002596,0.000064,0.000075
min,-0.006647,-0.000181,0.000047
25%,-0.001485,0.000091,0.000101
50%,-0.000246,0.000110,0.000124
75%,0.001451,0.000148,0.000163
max,0.015161,0.000759,0.000725


In [40]:
combo1_ext[["RR_D1","RR_W1","RR_M1"]].describe()

,RR_D1,RR_W1,RR_M1
count,3710.000000,3710.000000,3710.000000
mean,-0.000183,-0.000221,-0.000400
std,0.000336,0.000390,0.000525
min,-0.002460,-0.002520,-0.002641
25%,-0.000320,-0.000389,-0.000658
50%,-0.000142,-0.000181,-0.000315
75%,0.000014,0.000019,-0.000035
max,0.001726,0.001857,0.001824


In [41]:
combo1_ext.loc[combo1_ext["BF_D1"].abs() > 0.005]

,date,close,log_return,log_return_unshifted,return,return_unshifted,RV,RV_W,RV_M,CC,...,IV_CURVE_D1_W1_M1,IV_CURVE_W1_M1_M3,IV_CURVE_M1_M6_Y1,IV_CURVE_M3_M6_Y1,RR_D1,RR_W1,RR_M1,BF_D1,BF_W1,BF_M1
156,2008-09-15,1.417805,-0.001993,0.000497,-0.001991,0.000497,0.000177,0.000118,0.000063,0.000159,...,-0.002324,0.001450,0.001240,0.000168,-0.000365,-0.000351,-0.000419,0.005183,0.000129,0.000150
159,2008-09-18,1.438035,0.000104,0.017612,0.000104,0.017768,0.000160,0.000160,0.000080,0.000136,...,-0.001937,0.001083,0.001342,0.000293,0.000042,0.000050,0.000072,0.005177,0.000142,0.000159
166,2008-09-29,1.437920,-0.021198,-0.014893,-0.020975,-0.014783,0.000080,0.000091,0.000100,0.000072,...,-0.002746,0.001644,0.001489,0.000221,-0.000181,-0.000183,-0.000113,0.006641,0.000146,0.000180
167,2008-09-30,1.407760,-0.001375,-0.021198,-0.001375,-0.020975,0.000171,0.000103,0.000106,0.000172,...,-0.003140,0.001737,0.001928,0.000343,-0.000308,-0.000329,-0.000251,0.007824,0.000170,0.000202
168,2008-10-01,1.405825,-0.014672,-0.001375,-0.014564,-0.001375,0.000138,0.000112,0.000111,0.000144,...,-0.002957,0.001513,0.001928,0.000303,-0.000366,-0.000387,-0.000289,0.007674,0.000174,0.000211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3675,2022-11-03,0.975805,0.013932,-0.011858,0.014029,-0.011788,0.000127,0.000053,0.000059,0.000095,...,-0.000309,0.001171,-0.000145,0.000012,-0.000135,-0.000239,-0.000676,0.005286,0.000088,0.000128
3679,2022-11-09,1.002785,0.029308,-0.003872,0.029742,-0.003864,0.000051,0.000067,0.000058,0.000050,...,0.003290,0.001196,0.000118,0.000065,-0.000063,-0.000159,-0.000551,0.009203,0.000080,0.000117
3682,2022-11-16,1.040367,-0.007184,0.007102,-0.007159,0.007127,0.000119,0.000064,0.000062,0.000100,...,0.001442,-0.001264,0.001102,0.000080,-0.000109,-0.000227,-0.000553,0.005142,0.000101,0.000139
3692,2022-12-01,1.048720,-0.001317,0.018164,-0.001316,0.018330,0.000092,0.000051,0.000052,0.000094,...,0.002937,-0.000006,0.000718,0.000129,-0.000019,-0.000087,-0.000423,0.007622,0.000093,0.000115
